# gridMET — 4 km Daily Weather, 1979→Yesterday (built for agriculture)

**What it is.** Daily gridded surface meteorology for the CONUS at ~4 km, blending PRISM's
spatial detail with reanalysis. Includes the variables ag/hydrology models want, including
**reference ET** and **VPD**.

| | |
|---|---|
| **Coverage** | Contiguous U.S., ~4 km grid |
| **History** | 1979 → yesterday (daily, ~1-day latency) |
| **Cost / key** | **Free · no key** (OPeNDAP / THREDDS, Climatology Lab) |
| **Docs** | https://www.climatologylab.org/gridmet.html |

**Why it's core to Tracera.** A single, consistent, gap-free daily weather history for any
field — ideal for **Growing Degree Days**, water balance, and stress indices. More
model-ready than raw station data.

> **Access note.** We read via **OPeNDAP with `xarray`+`pydap`** (pure-Python, works on
> ARM64). This automatically applies the files' `scale_factor`/`add_offset` — the plain
> NCSS CSV returns *packed* values (e.g. 775 for temperature), so OPeNDAP is the correct path.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
from data_sources._common import FIELDS, get_key, field_polygon

# FIELDS is a LIST of field dicts. Loop over it for multi-field, or grab one:
#   FIELDS[0]["lat"]  ->  a single value    (NOT FIELDS["lat"] — that's a TypeError)
PRIMARY = FIELDS[0]
LAT, LON = PRIMARY["lat"], PRIMARY["lon"]
FIPS, STATE, COUNTY = PRIMARY["fips"], PRIMARY["state_alpha"], PRIMARY["county_name"]

print(f"Repo root: {REPO_ROOT}")
print(f"{len(FIELDS)} field(s) loaded:")
for f in FIELDS:
    print(f"  {f['id']} — {f['name']}: {f['acres']} ac {f['crop']} | "
          f"{f['county_name']} Co, {f['state_alpha']} | ({f['lat']}, {f['lon']})")

NameError: name 'REPO_ROOT' is not defined

### 1. Connection test — daily max temperature at the field (OPeNDAP)

In [2]:
import xarray as xr

# One aggregated NetCDF per variable. (var file token -> internal grid variable name)
GRIDMET = {
    "tmmx": "daily_maximum_temperature",                    # K
    "tmmn": "daily_minimum_temperature",                    # K
    "pr":   "precipitation_amount",                         # mm
    "pet":  "daily_mean_reference_evapotranspiration_grass",# mm (grass reference ET, ETo)
    "vpd":  "daily_mean_vapor_pressure_deficit",            # kPa
}
THREDDS = "http://thredds.northwestknowledge.net:8080/thredds/dodsC/agg_met_{tok}_1979_CurrentYear_CONUS.nc"

def gridmet_series(tok, start, end, lat=LAT, lon=LON):
    ds = xr.open_dataset(THREDDS.format(tok=tok), engine="pydap")
    var = GRIDMET[tok]
    s = ds[var].sel(lat=lat, lon=lon, method="nearest").sel(day=slice(start, end))
    return s.to_series()

tmax = gridmet_series("tmmx", "2023-07-01", "2023-07-07")
print("Daily max temperature, first week of July 2023:")
(tmax - 273.15).round(1).rename("tmax_C")

C:\Users\kylep\AppData\Local\Programs\Python\Python312-arm64\Lib\site-packages\pydap\handlers\dap.py:164: UserWarning: PyDAP was unable to determine the DAP protocol defaulting to DAP2. DAP2 is consider legacy and may result in slower responses. 
Consider replacing `http` in your `url` with either `dap2` or `dap4` to specify the DAP protocol (e.g. `dap2://<data_url>` or `dap4://<data_url>`).  For more 
information, go to https://www.opendap.org/faq-page.
  warnings.warn(


Daily max temperature, first week of July 2023:


day
2023-07-01    24.4
2023-07-02    29.9
2023-07-03    31.0
2023-07-04    31.7
2023-07-05    25.6
2023-07-06    23.2
2023-07-07    20.2
Name: tmax_C, dtype: float64

### 2. Core query — multi-variable daily table + Growing Degree Days

In [3]:
start, end = "2023-05-01", "2023-09-30"
wx = pd.DataFrame({tok: gridmet_series(tok, start, end) for tok in GRIDMET})
wx = wx.rename(columns={"tmmx": "tmax_K", "tmmn": "tmin_K", "pr": "precip_mm",
                        "pet": "ref_ET_mm", "vpd": "vpd_kPa"})
wx["tmax_C"] = wx["tmax_K"] - 273.15
wx["tmin_C"] = wx["tmin_K"] - 273.15
# Corn GDD (base 10C, cap 30C) — a core agronomic accumulator
tmax_c = wx["tmax_C"].clip(upper=30)
tmin_c = wx["tmin_C"].clip(lower=10)
wx["GDD_corn"] = (((tmax_c + tmin_c) / 2) - 10).clip(lower=0)
print(f"Season GDD (corn, base 10C): {wx['GDD_corn'].sum():.0f}   "
      f"Total precip: {wx['precip_mm'].sum():.0f} mm   Ref ET: {wx['ref_ET_mm'].sum():.0f} mm")
wx[["tmax_C", "tmin_C", "precip_mm", "ref_ET_mm", "vpd_kPa", "GDD_corn"]].head()

C:\Users\kylep\AppData\Local\Programs\Python\Python312-arm64\Lib\site-packages\pydap\handlers\dap.py:164: UserWarning: PyDAP was unable to determine the DAP protocol defaulting to DAP2. DAP2 is consider legacy and may result in slower responses. 
Consider replacing `http` in your `url` with either `dap2` or `dap4` to specify the DAP protocol (e.g. `dap2://<data_url>` or `dap4://<data_url>`).  For more 
information, go to https://www.opendap.org/faq-page.
  warnings.warn(


C:\Users\kylep\AppData\Local\Programs\Python\Python312-arm64\Lib\site-packages\pydap\handlers\dap.py:164: UserWarning: PyDAP was unable to determine the DAP protocol defaulting to DAP2. DAP2 is consider legacy and may result in slower responses. 
Consider replacing `http` in your `url` with either `dap2` or `dap4` to specify the DAP protocol (e.g. `dap2://<data_url>` or `dap4://<data_url>`).  For more 
information, go to https://www.opendap.org/faq-page.
  warnings.warn(


C:\Users\kylep\AppData\Local\Programs\Python\Python312-arm64\Lib\site-packages\pydap\handlers\dap.py:164: UserWarning: PyDAP was unable to determine the DAP protocol defaulting to DAP2. DAP2 is consider legacy and may result in slower responses. 
Consider replacing `http` in your `url` with either `dap2` or `dap4` to specify the DAP protocol (e.g. `dap2://<data_url>` or `dap4://<data_url>`).  For more 
information, go to https://www.opendap.org/faq-page.
  warnings.warn(


C:\Users\kylep\AppData\Local\Programs\Python\Python312-arm64\Lib\site-packages\pydap\handlers\dap.py:164: UserWarning: PyDAP was unable to determine the DAP protocol defaulting to DAP2. DAP2 is consider legacy and may result in slower responses. 
Consider replacing `http` in your `url` with either `dap2` or `dap4` to specify the DAP protocol (e.g. `dap2://<data_url>` or `dap4://<data_url>`).  For more 
information, go to https://www.opendap.org/faq-page.
  warnings.warn(


C:\Users\kylep\AppData\Local\Programs\Python\Python312-arm64\Lib\site-packages\pydap\handlers\dap.py:164: UserWarning: PyDAP was unable to determine the DAP protocol defaulting to DAP2. DAP2 is consider legacy and may result in slower responses. 
Consider replacing `http` in your `url` with either `dap2` or `dap4` to specify the DAP protocol (e.g. `dap2://<data_url>` or `dap4://<data_url>`).  For more 
information, go to https://www.opendap.org/faq-page.
  warnings.warn(


Season GDD (corn, base 10C): 1647   Total precip: 479 mm   Ref ET: 803 mm


,tmax_C,tmin_C,precip_mm,ref_ET_mm,vpd_kPa,GDD_corn
day,,,,,,
2023-05-01,15.15,4.05,0.0,5.2,0.80,2.575
2023-05-02,17.05,0.15,0.0,5.3,0.87,3.525
2023-05-03,21.55,3.65,0.0,4.6,1.21,5.775
2023-05-04,26.25,9.65,0.0,6.7,1.56,8.125
2023-05-05,21.05,13.25,29.2,4.3,0.77,7.150


### Notes & how to extend
- **All variables** (file token): `tmmx`/`tmmn` (temp K), `pr` (precip mm), `pet`/`etr`
  (grass/alfalfa reference ET mm), `vpd` (kPa), `srad` (W/m²), `rmax`/`rmin` (RH %),
  `vs` (wind m/s), `sph` (specific humidity).
- **Water balance:** `precip_mm − ref_ET_mm` per day ≈ surplus/deficit; cumulative deficit
  flags irrigation need. Cross-check against **OpenET** actual ET.
- **Port note:** use `:8080` (http). The `:8443` (https) endpoint may be blocked by some
  networks (it is on this machine).
- **County/region:** replace `.sel(lat, lon)` with a bounding-box `.sel(lat=slice(...),
  lon=slice(...))` and average.